# Dynamic Auto-Tagging with RAG-based Few-Shot Learning

This notebook demonstrates how to use dynamic prompting with TF-IDF retrieval for ticket classification.

In [ ]:
import sys
import pandas as pd
sys.path.append('..')

from utils.label_creation_dynamic import DynamicAutoTagger

## 1. Initialize the Dynamic Tagger

The tagger will:
- Load the example database (prediction_results.csv)
- Build TF-IDF vectors for all examples
- Ready to retrieve similar examples dynamically

In [ ]:
# Initialize with config
tagger = DynamicAutoTagger(
    config_path='../config/config_labels_dynamic.yaml',
    example_db_path='../data/prediction_results.csv',  # Your labeled examples
    max_workers=5  # Parallel processing
)

# Test connection
tagger.test_connection()

## 2. Check Retrieval Stats

In [ ]:
# See how many examples are loaded
stats = tagger.get_retrieval_stats()
print(f"Examples in database: {stats['num_examples_in_db']}")
print(f"Retrieval method: {stats['retrieval_method']}")
print(f"Examples per query: {stats['num_examples_per_query']}")
print(f"Min similarity: {stats['min_similarity_threshold']}")

## 3. Test Retrieval on a Single Example

Let's see what examples get retrieved for a test query

In [ ]:
# Test query
test_text = "Customer cannot login to YouSee Play app. Getting error message."

# Retrieve similar examples
retrieved = tagger.retrieve_examples(test_text)

print(f"Retrieved {len(retrieved)} examples:\n")
for i, ex in enumerate(retrieved, 1):
    print(f"Example {i} (similarity: {ex['similarity']:.3f})")
    print(f"Text: {ex['text'][:100]}...")
    print(f"Label: {ex['label']}")
    print()

## 4. Load Your Data to Classify

In [ ]:
# Load your tickets
df = pd.read_csv('../data/combined_data_masked.csv')

# Create text column (combine title + description + assignment_group)
df['text'] = df['title'] + ' ' + df['description'] + ' ' + df['assignment_group']

# Select columns for prediction
prediction_df = df[["number", "text"]].copy()

print(f"Loaded {len(prediction_df)} tickets to classify")
prediction_df.head()

## 5. Run Predictions with Dynamic Few-Shot Examples

For each ticket, the system will:
1. Retrieve 3 most similar examples from the database
2. Build a dynamic prompt with those examples
3. Get predictions from the LLM
4. Store predictions + the retrieved examples

In [ ]:
# Run on small test set first
test_df = prediction_df.head(10)

results = tagger.predict_and_create_csv(
    df=test_df,
    output_path='../data/dynamic_predictions_test.csv',
    number_column='number',
    text_column='text'
)

## 6. Examine Results

In [ ]:
# Load results
results = pd.read_csv('../data/dynamic_predictions_test.csv')

print(f"Total rows: {len(results)}")
print(f"Unique tickets: {results['number'].nunique()}")
print(f"\nColumns: {results.columns.tolist()}")

# Show first ticket (3 predictions)
first_ticket = results[results['number'] == results['number'].iloc[0]]
print(f"\nFirst ticket predictions:")
print(first_ticket[['number', 'label', 'confidence_score', 'shot1', 'shot2', 'shot3']].to_string(index=False))

## 7. Analyze Retrieved Examples

Let's see what examples were retrieved for each prediction

In [ ]:
# Pick one ticket to examine
ticket_num = results['number'].iloc[0]
ticket_data = results[results['number'] == ticket_num].iloc[0]

print(f"Ticket: {ticket_num}")
print(f"\nText: {ticket_data['text'][:200]}...\n")
print(f"Retrieved Examples:")
print(f"\nShot 1: {ticket_data['shot1']}")
print(f"\nShot 2: {ticket_data['shot2']}")
print(f"\nShot 3: {ticket_data['shot3']}")
print(f"\nPredicted Labels:")
for i, row in results[results['number'] == ticket_num].iterrows():
    print(f"  Top{i%3+1}: {row['label']} (confidence: {row['confidence_score']:.3f})")

## 8. Run on Full Dataset

In [ ]:
# Uncomment to run on full dataset
# results_full = tagger.predict_and_create_csv(
#     df=prediction_df,
#     output_path='../data/dynamic_predictions_full.csv',
#     number_column='number',
#     text_column='text'
# )

## 9. Compare Static vs Dynamic Prompting

In [ ]:
# Load static predictions (if you have them)
# static_results = pd.read_csv('../data/prediction_results.csv')
# dynamic_results = pd.read_csv('../data/dynamic_predictions_full.csv')

# # Compare top1 accuracy or other metrics
# # Add your comparison analysis here

## Key Differences from Static Prompting:

### Static Prompting:
- Uses same 7 hardcoded examples for ALL tickets
- Examples may not be relevant to the input

### Dynamic Prompting (This Notebook):
- Retrieves 3 most similar examples for EACH ticket
- Examples are always relevant to the input
- Stores which examples were used (shot1, shot2, shot3)
- Can analyze which examples lead to better predictions

### Output Format:
```
number | text | label | reasoning | confidence_score | shot1 | shot2 | shot3
```

Each ticket gets 3 rows (top3 predictions), all with the same retrieved examples.